# From x86-64 to TinyCPU: how a processor runs a program

## 1. Machine instructions

**Machine code** consists of bytes a processor understands; **assembly** is a readable representation of instructions. An **opcode** specifies the action; **operands** identify its inputs and destination. In Intel syntax, `mov rax, 5` uses an immediate (a number in the instruction), `add rax, rbx` uses registers, and `mov rax, [rsp]` reads memory at an address. `mov [rsp], rax` writes memory.

x86-64 instructions vary in byte length. `RIP` holds the address of the next instruction, so it advances by the current instruction's length or changes to a jump target. The architecture specifies visible behavior; a modern processor's internal implementation can differ substantially.

**Question:** How do `mov rax, 5` and `mov rax, [5]` differ?

## 2. Fetch, decode, execute

A simplified execution cycle: (1) fetch bytes at `RIP`; (2) decode their length and operation; (3) obtain operands; (4) execute; (5) store the result and update flags; (6) choose the next `RIP`. Modern x86 processors use pipelines, branch prediction, and micro-operations. This cycle is an architectural model, not a literal account of one hardware clock cycle.

## 3. Registers

General-purpose registers (`RAX`, `RBX`, etc.) hold data. `RSP` typically points to the top of the stack. `RIP` controls instruction flow. `RFLAGS` records properties of results. A conceptual instruction register holds the instruction being decoded, but it is not an ordinary programmer-accessible x86-64 register.

**Question:** Why do `RSP` and `RIP` have special roles?

## 4. RFLAGS: four essential flags

| Flag | Meaning | Typical use |
|---|---|---|
| `ZF` | The result is zero | `JE`/`JZ` |
| `SF` | The high sign bit is set | Result sign |
| `CF` | Unsigned carry or borrow | Unsigned comparison |
| `OF` | Signed overflow | Signed comparison |

`cmp a, b` sets flags as though it computed `a-b`, but does not store that result. For signed `a<b`, test `SF != OF`: overflow can make `SF` alone misleading. In 32 bits, `-2147483648 - 1` produces `2147483647`, with `SF=0` and `OF=1`, so `JL` is true. For unsigned `a<b`, test `CF` after subtraction.

| Jump | Condition |
|---|---|
| `JE` | `ZF` |
| `JNE` | `!ZF` |
| `JL` | `SF!=OF` |
| `JLE` | `ZF || SF!=OF` |
| `JG` | `!ZF && SF==OF` |
| `JGE` | `SF==OF` |

`2147483647 + 1` overflows signed 32-bit arithmetic: the result is `-2147483648` and `OF=1`. Different instructions update different subsets of flags; `MOV` leaves all flags unchanged in TinyCPU.

**Question:** What is `ZF` after `CMP R0, R0`?

## 5. Branches and loops

#### Condition:
```python
if a < b:
    result=a
else:
    result=b
```
is converted to:
```asm
CMP a, b
JL then
MOV result, b
JMP end
then:
MOV result, a
end:
...
```


#### Loops

```python
while n > 0:
    total += n
    n -= 1
```
is converted to:
```asm
loop:
CMP n, 0
JLE done
ADD total, n
SUB n, 1
JMP loop
done:
...
```

```python
for i in range(1, n + 1):
    total += i
```

is converted to:
```asm
MOV i, 1
loop:
CMP i, n
JG done
ADD total, i
ADD i, 1
JMP loop
done:
...
```

There are no separate `if`, `while`, or `for` instructions. State, comparisons, and jumps implement them.

## 6. TinyCPU

TinyCPU has four registers, a single 32-bit integer format, and 256 memory addresses. One instruction occupies one `PC` position; there is no byte encoding. `INPUT=240` and `OUTPUT=241` implement memory-mapped I/O: devices are accessed through special memory addresses. Memory may be addressed directly as `[10]` or indirectly through a register as `[R2]`; the latter uses the value currently in `R2` as the address. There are no `MUL`, `DIV`, or `MOD` instructions; students build those operations from addition, subtraction, and loops. See the [architecture specification](../docs/architecture.md).

In [5]:
from pathlib import Path
import sys
try:
    from tinycpu import assemble, TinyCPU
except (ModuleNotFoundError, ImportError):
    root = Path.cwd()
    if (root / 'TinyCpu' / 'src' / 'tinycpu').is_dir():
        sys.path.insert(0, str(root / 'TinyCpu' / 'src'))
    elif (root.parent / 'src' / 'tinycpu').is_dir():
        sys.path.insert(0, str(root.parent / 'src'))
    else:
        raise RuntimeError('Open the notebook from the tinycpu root or install the package')
    from tinycpu import assemble, TinyCPU
print('TinyCPU is ready')

TinyCPU is ready


#### Notice, TinyCPU works if you clone the repo locally. If you need it in GoogleColab, please, let me know.

## 7. Live programs

First, add two numbers read from the input port. Then inspect a step-by-step table with the step number, `PC`, instruction, registers, and flags. The table is compact enough for a projector.

In [ ]:
sum_source = """LOAD R0, [INPUT]
LOAD R1, [INPUT]
ADD R0, R1
STORE [OUTPUT], R0
HALT"""
print('Sum:', TinyCPU([2,3]).run(assemble(sum_source)).outputs)

In [ ]:
cpu = TinyCPU([2,3]); cpu.load(assemble(sum_source))
print('step | PC | instruction            | R0 R1 R2 R3 | Z S C O')
while not cpu.halted:
    s = cpu.step()
    print(f'{s.number:>4} | {s.pc:>2} | {s.instruction.source:<22} | {str(s.registers):<13} | {tuple(map(int,s.flags))}')

Next, compute `max(a,b)`: `CMP` sets flags without storing a difference, and `JGE` chooses a branch.

In [ ]:
max_source = """LOAD R0, [INPUT]
LOAD R1, [INPUT]
CMP R0, R1
JGE done
MOV R0, R1
done: STORE [OUTPUT], R0
HALT"""
for pair in ((7,3),(3,7)):
    cpu=TinyCPU(pair); cpu.load(assemble(max_source)); cpu.run()
    print(pair,'→',cpu.outputs,'flags',cpu.flags)

The sum from `1` through `N` illustrates a loop: `JMP loop` returns execution to the comparison.

In [ ]:
loop_source = """LOAD R0, [INPUT]
MOV R1, 0
loop: CMP R0, 0
JLE done
ADD R1, R0
SUB R0, 1
JMP loop
done: STORE [OUTPUT], R1
HALT"""
for n in (0,1,5):
    cpu=TinyCPU([n]); print(n,'→',cpu.run(assemble(loop_source)).outputs,'steps',cpu.steps)

The assembler reports the source line, and the instruction limit stops an infinite loop.

In [ ]:
from tinycpu.errors import TinyCPUError
for source, limit in [('MOV R4, 1\nHALT', 10), ('loop: JMP loop', 4)]:
    try:
        TinyCPU(instruction_limit=limit).run(assemble(source))
    except TinyCPUError as error:
        print(error)

### Review questions

1. What `PC` follows a taken `JMP loop`?
2. Which flags are set by `CMP R0, 0` when `R0=0`?
3. How many times does the sum loop body run for `N=5`?
4. What appears at `OUTPUT` for input `7 3` in the `max` program?
5. What is wrong with `LOAD R0, OUTPUT`?